In [1]:
# Load the processed dataset for feature engineering
from pathlib import Path

import pandas as pd
import numpy as np

project_root = Path.cwd().parent
data_path = project_root / "data" / "processed" / "model_data.csv"

df = pd.read_csv(data_path, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

print("Shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())

df.head()

Shape: (3607, 9)
Date range: 2016-11-09 00:00:00 to 2026-09-24 00:00:00


,date,pm25,temperature,humidity,rainfall,wind_speed,fire_count,total_frp,mean_frp
0,2016-11-09,270.0,21.7,51,0.0,9.5,4052.0,32329.46,7.978643
1,2016-11-10,259.0,21.0,59,0.0,7.1,3169.0,17701.61,5.585866
2,2016-11-11,274.0,21.0,64,0.0,7.4,4339.0,32915.66,7.586001
3,2016-11-12,238.0,21.7,60,0.0,7.6,2280.0,15017.25,6.586513
4,2016-11-13,175.0,21.9,58,0.0,4.9,1195.0,7370.45,6.167741


In [2]:
# Create calendar features to capture seasonal pollution patterns
df["month"] = df["date"].dt.month
df["day_of_year"] = df["date"].dt.dayofyear

df["season"] = df["month"].map({
    12: "winter", 1: "winter", 2: "winter",
    3: "summer", 4: "summer", 5: "summer", 6: "summer",
    7: "monsoon", 8: "monsoon", 9: "monsoon",
    10: "post_monsoon", 11: "post_monsoon"
})

df[["date", "month", "day_of_year", "season"]].head()

,date,month,day_of_year,season
0,2016-11-09,11,314,post_monsoon
1,2016-11-10,11,315,post_monsoon
2,2016-11-11,11,316,post_monsoon
3,2016-11-12,11,317,post_monsoon
4,2016-11-13,11,318,post_monsoon


In [3]:
# Create lag features using previous PM2.5 observations
df["pm25_lag_1"] = df["pm25"].shift(1)
df["pm25_lag_2"] = df["pm25"].shift(2)
df["pm25_lag_3"] = df["pm25"].shift(3)
df["pm25_lag_7"] = df["pm25"].shift(7)

df[
    ["date", "pm25", "pm25_lag_1", "pm25_lag_2", "pm25_lag_3", "pm25_lag_7"]
].head(10)

,date,pm25,pm25_lag_1,pm25_lag_2,pm25_lag_3,pm25_lag_7
0,2016-11-09,270.0,NaN,NaN,NaN,NaN
1,2016-11-10,259.0,270.0,NaN,NaN,NaN
2,2016-11-11,274.0,259.0,270.0,NaN,NaN
3,2016-11-12,238.0,274.0,259.0,270.0,NaN
4,2016-11-13,175.0,238.0,274.0,259.0,NaN
5,2016-11-14,167.0,175.0,238.0,274.0,NaN
6,2016-11-15,202.0,167.0,175.0,238.0,NaN
7,2016-11-16,250.0,202.0,167.0,175.0,270.0
8,2016-11-17,215.0,250.0,202.0,167.0,259.0
9,2016-11-18,195.0,215.0,250.0,202.0,274.0


In [4]:
# Create recent pollution trend features without using today's PM2.5
previous_pm25 = df["pm25"].shift(1)

df["pm25_roll_3"] = previous_pm25.rolling(
    window=3,
    min_periods=1
).mean()

df["pm25_roll_7"] = previous_pm25.rolling(
    window=7,
    min_periods=1
).mean()

df[
    ["date", "pm25", "pm25_lag_1", "pm25_roll_3", "pm25_roll_7"]
].head(10)

,date,pm25,pm25_lag_1,pm25_roll_3,pm25_roll_7
0,2016-11-09,270.0,NaN,NaN,NaN
1,2016-11-10,259.0,270.0,270.000000,270.000000
2,2016-11-11,274.0,259.0,264.500000,264.500000
3,2016-11-12,238.0,274.0,267.666667,267.666667
4,2016-11-13,175.0,238.0,257.000000,260.250000
5,2016-11-14,167.0,175.0,229.000000,243.200000
6,2016-11-15,202.0,167.0,193.333333,230.500000
7,2016-11-16,250.0,202.0,181.333333,226.428571
8,2016-11-17,215.0,250.0,206.333333,223.571429
9,2016-11-18,195.0,215.0,222.333333,217.285714


In [5]:
# Create the target: PM2.5 value for the next day
df["target_pm25_next_day"] = df["pm25"].shift(-1)

df[
    ["date", "pm25", "target_pm25_next_day"]
].head()

,date,pm25,target_pm25_next_day
0,2016-11-09,270.0,259.0
1,2016-11-10,259.0,274.0
2,2016-11-11,274.0,238.0
3,2016-11-12,238.0,175.0
4,2016-11-13,175.0,167.0


In [6]:
# Check missing values in the features and prediction target
feature_columns = [
    "temperature", "humidity", "rainfall", "wind_speed",
    "fire_count", "total_frp", "mean_frp",
    "month", "day_of_year",
    "pm25_lag_1", "pm25_lag_2", "pm25_lag_3", "pm25_lag_7",
    "pm25_roll_3", "pm25_roll_7"
]

print("Total rows:", len(df))

print("\nMissing feature values:")
print(df[feature_columns].isna().sum())

print("\nMissing target values:")
print(df["target_pm25_next_day"].isna().sum())

Total rows: 3607

Missing feature values:
temperature       0
humidity          0
rainfall          0
wind_speed        0
fire_count     1012
total_frp      1012
mean_frp       1012
month             0
day_of_year       0
pm25_lag_1      152
pm25_lag_2      153
pm25_lag_3      154
pm25_lag_7      158
pm25_roll_3     127
pm25_roll_7      94
dtype: int64

Missing target values:
152


In [7]:
# Keep only rows with complete information needed for model training
model_features = feature_columns + ["season"]

ml_df = df.dropna(
    subset=feature_columns + ["target_pm25_next_day"]
).copy()

print("Original rows:", len(df))
print("ML-ready rows:", len(ml_df))
print("Date range:", ml_df["date"].min(), "to", ml_df["date"].max())
print("Remaining missing values:", ml_df[feature_columns + ["target_pm25_next_day"]].isna().sum().sum())

Original rows: 3607
ML-ready rows: 2367
Date range: 2016-11-16 00:00:00 to 2023-12-17 00:00:00
Remaining missing values: 0


In [8]:
# Save the final feature-engineered dataset for model experiments
feature_path = project_root / "data" / "processed" / "features.csv"

ml_df.to_csv(feature_path, index=False)

print("Saved:", feature_path)
print("Shape:", ml_df.shape)

ml_df.head()

Saved: c:\Users\aayus\OneDrive\Desktop\Data Science Project\Air Quality Trend & Forecast\data\processed\features.csv
Shape: (2367, 19)


,date,pm25,temperature,humidity,rainfall,wind_speed,fire_count,total_frp,mean_frp,month,day_of_year,season,pm25_lag_1,pm25_lag_2,pm25_lag_3,pm25_lag_7,pm25_roll_3,pm25_roll_7,target_pm25_next_day
7,2016-11-16,250.0,20.1,52,0.0,8.7,1180.0,8069.43,6.838500,11,321,post_monsoon,202.0,167.0,175.0,270.0,181.333333,226.428571,215.0
8,2016-11-17,215.0,20.1,50,0.0,12.5,1335.0,7768.52,5.819116,11,322,post_monsoon,250.0,202.0,167.0,259.0,206.333333,223.571429,195.0
9,2016-11-18,195.0,20.1,55,0.0,10.4,915.0,4957.63,5.418175,11,323,post_monsoon,215.0,250.0,202.0,274.0,222.333333,217.285714,201.0
10,2016-11-19,201.0,19.6,58,0.0,7.0,398.0,2239.32,5.626432,11,324,post_monsoon,195.0,215.0,250.0,238.0,220.000000,206.000000,211.0
11,2016-11-20,211.0,19.7,62,0.0,6.4,453.0,2635.18,5.817174,11,325,post_monsoon,201.0,195.0,215.0,175.0,203.666667,200.714286,198.0
